In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

In [ ]:
spg_list = [1, 2, 3, 9, 14, 19, 33]   

def plot_energy_density_by_spg(csv_path, energy_offset=0.0):
    df = pd.read_csv(csv_path)

    rows = []

    for block_index, spg in enumerate(spg_list):
        c0 = df.columns[block_index*3 + 1]
        c1 = df.columns[block_index*3 + 2]
        c2 = df.columns[block_index*3 + 3]

        for i in range(len(df)):
            rows.append({
                "idx": df[c0].iloc[i],
                "energy": ((df[c1].iloc[i])/4 - energy_offset)*96.5,  # in kj per mol
                "density": df[c2].iloc[i],
                "spg": str(spg)
            })

    df_long = pd.DataFrame(rows)

    fig = px.scatter(
        df_long,
        x="density",
        y="energy",
        color="spg",
        hover_data=["idx", "spg"],
        title=f"{csv_path.strip('.csv')} (Energy vs Density)",
        color_discrete_sequence=px.colors.qualitative.Pastel
    )

    fig.update_traces(marker=dict(size=10))

    fig.update_layout(
        xaxis_title="Density / g cm⁻³",
        yaxis_title=f"Cohesive energy kJ mol⁻¹" if energy_offset != 0 else "Cohesive energy / kJ mol⁻¹",

        plot_bgcolor="white",
        paper_bgcolor="white",

        xaxis=dict(
            showline=True,
            linecolor="black",
            mirror=True,
            ticks="outside",
            tickfont=dict(size=16),
            showgrid=False
        ),
        yaxis=dict(
            showline=True,
            linecolor="black",
            mirror=True,
            ticks="outside",
            tickfont=dict(size=16),
            showgrid=False
        ),

        legend_title="spg",
        margin=dict(l=60, r=60, t=60, b=60)
    )

    fig.show()
    fig.write_html(f"{csv_path.strip('.csv')}_EnergyDensity_spg.html")

In [3]:
# csvs = {'SO3_Na_all_sg_uff.csv': 100.4 , 'SO3_NMe4_all_sg_uff.csv': (100.4 + 5*4),'Me_SO4_all_sg_uff.csv': (45.91 + 4.5*4)} 
csvs = {'SO3_Na_all_sg_uff.csv': 0 , 'SO3_NMe4_all_sg_uff.csv': 0,'Me_SO4_all_sg_uff.csv': 0} 

In [4]:
for csv in csvs.keys():
    plot_energy_density_by_spg(csv, energy_offset=csvs[csv])

In [106]:
def plot_histogram(csv_path, energy_offset=0.0, n_bins=200, x_type='energy', energy_max=0):
    """
    Plot histogram of energies or densities from a CSV file.

    x_type: 'energy' or 'density'
    energy_max: maximum energy to include in histogram (can be negative)
    
    The histogram is normalized so the tallest bin corresponds to 100.
    """
    df = pd.read_csv(csv_path)
    values = []

    if x_type == 'energy':
        for i in range(2, df.shape[1], 3):
            col = pd.to_numeric(df.iloc[:, i], errors='coerce').dropna()
            energies = ((col / 4 - energy_offset) * 96.5).tolist()

            if energy_max is not None:
                energies = [e for e in energies if e < energy_max]

            values.extend(energies)

        x_label = f"Cohesion Energy / kJ mol⁻¹ (<{energy_max})" if energy_max is not None else "Cohesion Energy / kJ mol⁻¹"

    elif x_type == 'density':
        for i in range(1, df.shape[1], 3):
            col = pd.to_numeric(df.iloc[:, i], errors='coerce').dropna()
            values.extend(col.tolist())
        x_label = "Density / g cm⁻³"

    else:
        raise ValueError("x_type must be 'energy' or 'density'")

    # Compute histogram
    counts, bins = np.histogram(values, bins=n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    # Normalize so the tallest bin = 100
    if len(counts) > 0:
        counts = counts / counts.max() * 100

    # Plot
    fig = px.bar(
        x=bin_centers,
        y=counts,
        title=f"{x_type.capitalize()} Histogram for {csv_path} (max bin normalized to 100)",
        labels={"x": x_label, "y": "Normalized Count (max = 100)"},
        color_discrete_sequence=['#636EFA']
    )
    fig.show()


In [114]:
csvs = {'SO3_Na_all_sg_uff.csv': 100.4 , 'SO3_NMe4_all_sg_uff.csv': (100.4 + 5*4),'Me_SO4_all_sg_uff.csv': (45.91 + 4.5*4)} 

In [118]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_density_histogram(csv_path, n_bins=50):
    """
    Plot a histogram of densities from a CSV file.
    The tallest bin is normalized to 100.
    """
    df = pd.read_csv(csv_path)
    densities = []

    # Assuming density is in the first column of each 3-column block
    for i in range(3, df.shape[1], 3):
        col = pd.to_numeric(df.iloc[:, i], errors='coerce').dropna()
        densities.extend(col.tolist())

    # Compute histogram
    counts, bins = np.histogram(densities, bins=n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    # Normalize so tallest bin = 100
    if len(counts) > 0:
        counts = counts / counts.max() * 100

    # Plot
    fig = px.bar(
        x=bin_centers,
        y=counts,
        title=f"Density Histogram for {csv_path}",
        labels={"x": "Density / g cm⁻³", "y": "Normalized Count (max=100)"},
        color_discrete_sequence=['#EF553B']
    )
    fig.update_yaxes(range=[0, 100])
    fig.show()

plot_density_histogram('SO3_Na_all_sg_uff.csv')
plot_density_histogram('SO3_NMe4_all_sg_uff.csv')
plot_density_histogram('Me_SO4_all_sg_uff.csv')


In [ ]:
for csv in csvs.keys():
    plot_histogram(csv, energy_offset=csvs[csv])